# NFL Survivor Pool Optimizer

Pick one team per week to win straight-up. Use each team at most once all season. One loss and you're out.

**The idea:** picking the highest win-probability team every week (greedy) is *not* optimal -- it can waste a team's "safety" on a week where it barely mattered, leaving worse options later. What you actually want is the set of picks (one team per week, no repeats) that maximizes the **product** of your win probabilities across the whole season.

Maximizing a product of probabilities = maximizing a sum of their logs. That turns this into a weighted bipartite matching problem (**the assignment problem**): weeks on one side, teams on the other, edge weight = `log(P(team wins that week))`. Solved exactly and quickly with the Hungarian algorithm (`scipy.optimize.linear_sum_assignment`) -- no greedy heuristics, no brute force.

**Two modes:**
- `survival` -- maximize your own probability of surviving every remaining week.
- `equity` -- maximize actual expected pot share, using real pot-splitting math. Value isn't just win probability, it's win probability diluted by how much of the field shares your pick. Derived directly (see the equity-mode section below), not a tunable heuristic. There's deliberately no pool-size-based caution mechanism -- see that section for why.

**Double-pick weeks:** some pools require picking TWO teams in certain weeks -- both must win, or you're out. Configure which weeks below (`DOUBLE_PICK_WEEKS`). A double-pick week is represented internally as two "slots" needing a team, both using that week's odds -- the assignment algorithm never assigns the same team to two slots, so it naturally finds the best two distinct teams on its own.

**Odds CSV format** (long form, two rows per game):
```
week,team,opponent,moneyline
1,BUF,ARI,-250
1,ARI,BUF,210
```

**Pick-popularity CSV format** (optional, for `equity` mode -- real data where you have it, e.g. from `fetch_pick_pcts.ipynb`; gaps get filled by the fallback model below):
```
week,team,pick_pct
1,BUF,0.34
```

**Availability CSV format** (optional, for `equity` mode -- current-week snapshot of what fraction of the field still has each team unused):
```
team,availability_pct
BUF,0.62
```

**Workflow:** run `fetch_odds.ipynb` to get `odds.csv`, and `fetch_pick_pcts.ipynb` to get `picks.csv` → run this notebook → act on the top pick(s) → set `LOCK` and re-run to advance state → repeat next week with fresh data.

## Imports

In [82]:
import argparse
import json
import math
import os
import sys
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

## Load Odds

Loads the odds CSV and de-vigs it (normalizes each game's two implied probabilities to sum to 1). Also guards against messy input: duplicate (week, team) rows get flagged with the exact conflicting values and resolved by keeping the last occurrence; any row left with a mismatched partner as a side-effect of that cleanup gets dropped rather than silently producing an impossible win probability.

In [83]:
def odds_to_prob(ml):
    # Converts American moneyline odds to implied probability
    ml = float(ml)
    if ml > 0:
        return 100.0 / (ml + 100.0)
    return -ml / (-ml + 100.0)


def load_odds(path):
    # Loads odds.csv file and converts moneyline probability to a devigged, true probability
    # of each team winning that specific game.
    
    # Loads odds.csv
    df = pd.read_csv(path)
    # Ensures teams and opponents are capitalized and do not have extra whitespace 
    df["team"] = df["team"].astype(str).str.upper().str.strip()
    df["opponent"] = df["opponent"].astype(str).str.upper().str.strip()

    # Convert moneyline odds to implied probability
    df["implied_prob"] = df["moneyline"].apply(odds_to_prob)

    # In order to devig the odds, a game key needs to be created to know which odds are part of the same
    # game. This chunk grabs each row of odds and creates a team/opponent tuple, which will act as the game
    # key when paired with the week of the game. Since that tuple is sorted, it will match for two different
    # sides of the same game, making the key unique. 
    game_keys = []
    for _, row in df.iterrows():
        sorted_teams = sorted([row["team"], row["opponent"]])
        game_keys.append((row["week"], tuple(sorted_teams)))
    df["game_key"] = game_keys

    # Devigs probability (the sportsbook's profit margin) to derive the true probability
    # of each team winning a game. After this, for a given game, both team's win probability 
    # will add up to 1.
    total_game_prob = df.groupby("game_key")["implied_prob"].transform("sum")
    df["win_prob"] = df["implied_prob"] / total_game_prob
    # Drop helper game key column
    return df.drop(columns="game_key")

## Load Pick Distribution

In [84]:
def load_pick_dist(path):
    # Loads projected_pick_distribution.csv
    df = pd.read_csv(path)
    # Ensures teams are capitalized and do not have extra whitespace 
    df["team"] = df["team"].astype(str).str.upper().str.strip()
    return df

## Load Team Availability

In [85]:
def load_team_availability(path):
    # Loads availability.csv
    df = pd.read_csv(path)
    # Ensures teams are capitalized and do not have extra whitespace 
    df["team"] = df["team"].astype(str).str.upper().str.strip()
    return df

## Load Survivor Pool State

In [86]:
def load_state(path, pool_name="default"):
    # Loads survivor_state.json, which can hold multiple pools' data at once,
    # and pulls the specifications for the pool requested. Returns the default
    # state if that pool is not found in the JSON file.

    # If no state file currently exists, return the default state, which assumes it is
    # Week 1 of the NFL season, no teams have been used, and there are no double pick weeks.
    default_state = {"used_teams": [], "current_week": 1, "double_pick_weeks": []}
    if not os.path.exists(path):
        return default_state

    # If a state file does exist, load it, including all pools contained within it
    with open(path) as file:
        all_pools = json.load(file)

    # If the request pool does not exist within the state file, print a warning message.
    # The optimizer will continue to operate as if the requested pool is in the default state.
    if all_pools and pool_name not in all_pools:
        print(f"Pool '{pool_name}' not found in {path}. "
              f"Existing pools in this file: {sorted(all_pools.keys())}. "
              f"Returning a default pool state... ")

    # Retrieves data for requested pool, returning the default state if that pool does not exist
    # within state file
    pool_data = all_pools.get(pool_name, {})
    return {
        "used_teams": pool_data.get("used_teams", []),
        "current_week": pool_data.get("current_week", 1),
        "double_pick_weeks": pool_data.get("double_pick_weeks", []),
    }

## Save Survivor Pool State

In [87]:
def save_state(path, pool_name, state):
   # Writes out new state of a survivor pool to survivor_state.json

    # If a state file already exists, load it
    all_pools = {}
    if os.path.exists(path):
        with open(path) as file:
            all_pools = json.load(file)
    # If the pool already exists within the state file, overwrite it with
    # the new data. If it does not, add the pool to the file.
    all_pools[pool_name] = state
    # Write out new state file
    with open(path, "w") as file:
        json.dump(all_pools, file, indent=2)

## Equity Mode

In [ ]:
def estimate_pick_distribution(odds_df, pick_pct_df, availability_df, herding_strength=8.0):
    # For any week in the future, the estimated pick distribution is not known. Typically, those
    # projections do not release until after the current week's games. So, if equity mode is selected 
    # (which is takes into account who others in the pool are selecting), it is necessary to have a 
    # mechanism to estimate what that pick distribution will be for future weeks. That is what this 
    # function does.

    # It does so by taking into account 3 factors: herding, availability, and future value. Herding refers to 
    # how much more often larger favorites are selected than smaller ones. In survivor pools, people often flock
    # to the largest of favorites exponentially more frequently. That is what this is capturing. Next, the 
    # availability of other teams is factored in. Once an entry uses a team, they cannot use them again. If a team
    # is not available to most of the field, the percentage of entries that pick them will inherently be smaller. 
    # Finally, it takes into account future value. If a team is a large favorite in the current week, but an underdog
    # or slight favorite in all other weeks, people are much more likely to pick them during the current week.
    # These 3 factors determine the estimated pick distribution for future weeks.

    # Obtains real pick distribution data, if it exists
    pick_lookup = {}
    if pick_pct_df is not None:
        for r in pick_pct_df.itertuples():
            pick_lookup[(r.week, r.team)] = r.proj_pick_dist

    # Obtains real team availability data, if it exists
    avail_lookup = {}
    if availability_df is not None:
        for r in availability_df.itertuples():
            avail_lookup[r.team] = r.availability_pct

    # Initializes values for each team with a dictionary of that team's win probability
    # for each week it plays
    team_week_probs = {}
    for row in odds_df.itertuples():
        if row.team not in team_week_probs:
            team_week_probs[row.team] = {}
        team_week_probs[row.team][row.week] = row.win_prob

    # Builds a dictionary that contains each team and their current availability within the pool
    # Updates as the solve progresses
    current_avail = {}
    for t in odds_df["team"].unique():
        current_avail[t] = avail_lookup.get(t, 1.0)


    rows = []
    # Loops through each week
    for week in sorted(odds_df["week"].unique()):
        # Filters to only that week's games
        weekly_games = odds_df[odds_df["week"] == week]
        # Keeps only necessary columns (team and win probability)
        weekly_teams = weekly_games[["team", "win_prob"]].drop_duplicates("team")
        # Adds real pick distribution data, if it exists
        known = {}
        for t in weekly_teams["team"]:
            if (week, t) in pick_lookup:
                known[t] = pick_lookup[(week, t)]
        # If there is no real pick distribution data for that week, all teams and win probabilities are assigned 
        # If there is real pick distribution data for that week, this will be empty
        missing = weekly_teams[~weekly_teams["team"].isin(known.keys())]
        estimated = {}
        # Only runs if there is no pick distribution data for that week
        if len(missing) > 0:
            raw_weights = {}
            # One team at a time
            for _, r in missing.iterrows():
                # Pulls team and win probability separately
                team, win_prob = r["team"], r["win_prob"]
                # Pulls team's current availability
                avail = current_avail.get(team, 1.0)
                # Pulls team's win probabilities for all other weeks
                other_probs = [v for w2, v in team_week_probs.get(team, {}).items() if w2 != week]

                # Calculates future value, which quantifies how often a team will have another week as a large favorite
                if other_probs:
                    # Determines average future win probability over remaining weeks
                    avg_future_win_prob = sum(other_probs) / len(other_probs)
                    # Quantifies future value by computing ratio of current week's win probability over total remaining win probability
                    # Asks "how much of future win probability is attributed to this week?" 
                    # If that ratio is large, that team does not have much future value outside of this week
                    # If that ratio is small, that team has many other weeks with a similar or higher win probability
                    future_value = win_prob / (win_prob + avg_future_win_prob) if (win_prob + avg_future_win_prob) > 0 else 1.0
                else:
                    # If this is the only remaining game, the future value ratio is 1
                    future_value = 1.0 
                # This combines the three factors into one metric: the current availability, the current week win probability 
                # (exponentiated by the herding strength, which adjusts for how much more valuable a team with a higher win 
                # probability is than one with a lower one), and the team's future value
                raw_weights[team] = avail * (win_prob ** herding_strength) * future_value

            # Adds each team's total
            weight_sum = sum(raw_weights.values())
            # Divides each team's score by the sum of all scores to determine their pick distribution share for that week
            if weight_sum > 0:
                for t, w in raw_weights.items():
                    estimated[t] = (w / weight_sum)

        week_shares = {}
        for t in weekly_teams["team"]:
            # Grabs team's real pick distribution first, but if that does not exist, then grab the estimated pick distribution
            share = known.get(t, estimated.get(t, 0.0))
            # Current week, team, and pick share is formatted to be converted to a DataFrame
            week_shares[t] = share
            rows.append({"week": week, "team": t, "pick_share": share})

        # Updates current availability to simulate team availability as if this pick distribution was real when calculating
        # the distribution for future weeks
        for t, share in week_shares.items():
            current_avail[t] = max(0.0, current_avail.get(t, 1.0) - share)

    return pd.DataFrame(rows)


def compute_field_survival(odds_df, pick_share_df, double_pick_weeks):
    # Calculates S (likelihood of survival) and K (number of correct picks needed)
    # for each week, which is used later on in the equity calculations.

    # Joins odds and projected pick distributions together
    merged = odds_df.merge(pick_share_df, on=["week", "team"], how="inner")
    out = {}
    for week, group in merged.groupby("week"):
        # For each week, multiplies each team's projected pick distribution percentage by
        # their win probability
        S = float((group["pick_share"] * group["win_prob"]).sum())
        # If a pick is a double pick week, 2 winners are needed to survive. Otherwise,
        # only 1 is needed
        out[week] = (S, 2 if week in double_pick_weeks else 1)
    return out


def project_pool_sizes(field_survival, start_week, pool_size):
    # Estimates the size of the pool after each week based on
    # dilution formula

    N = {}
    # Starts with actual remaining survivor pool size
    n = float(pool_size)
    for week in sorted(w for w in field_survival if w >= start_week):
        # Size of pool entering week
        N[week] = n
        S, K = field_survival[week]
        # Calculates estimated size of pool after week using dilution formula from above
        n *= S ** K
    return N


def build_equity_context(odds_df, state, pick_pct_df, availability_df, pool_size, herding_strength):
    # Puts together all context needed to calculate the equity portion of the project

    # Calculates projected pick distribution
    pick_share_df = estimate_pick_distribution(odds_df, pick_pct_df, availability_df, herding_strength)
    # Estimates survival probability for each week
    field_survival = compute_field_survival(odds_df, pick_share_df, set(state["double_pick_weeks"]))
    # Reshapes the projected pick distribution DataFrame
    pick_shares = {(r.week, r.team): r.pick_share for r in pick_share_df.itertuples()}
    # Combines survival probability and projected pick distribution
    ctx = {"field_survival": field_survival, "pick_shares": pick_shares}
    # Estimates the size of the pool after each week based on the survival probability and number of 
    # current entries
    if pool_size is not None:
        ctx["pool_sizes"] = project_pool_sizes(field_survival, state["current_week"], pool_size)
    return ctx

## Build Cost Matrix

In [ ]:
def build_cost_matrix(odds, state, mode, equity_ctx=None):
    # Filters out games that have already happened
    future_games = odds[odds["week"] >= state["current_week"]]

    # Sorts weeks and teams to be used as rows and columns in the matrix
    weeks = sorted(future_games["week"].unique())
    teams = sorted(future_games["team"].unique())

    # For all double pick weeks, adds an extra row in the matrix
    double_pick_weeks = set(state["double_pick_weeks"])
    week_rows = []
    for w in weeks:
        week_rows.append(w)
        if w in double_pick_weeks:
            week_rows.append(w)

    # Creates dictionary of each week and its row index in the matrix.
    # A week can contain a list of multiple indecies if it is a double pick week.
    week_index = {}
    for i, w in enumerate(week_rows):
        if w not in week_index:
            week_index[w] = []
        week_index[w].append(i)

    # Creates dictionary of each team and their column index in the matrix
    team_index = {}
    for i, t in enumerate(teams):
        team_index[t] = i

    # Initializes values within matrix with an extremely large number so the 
    # default behavior is that the team never gets picked
    cost = np.full((len(week_rows), len(teams)), 1e6)

    # Loops through each game in the matrix
    for row in future_games.itertuples():
        # Skips teams that have already been selected in a previous week
        if row.team in state["used_teams"]:
            continue
        # Gets week of game
        w = row.week
        # Gets column index for team
        t_i = team_index[row.team]
        # Ensures win probability is a value between 0 and 1.
        prob_bounds = min(max(row.win_prob, 1e-6), 1 - 1e-6)
        # Takes logarithm of win probability so they can be added during the solve.
        utility = math.log(prob_bounds)
        # If equity mode is selected, 
        if mode == "equity":
            # Retrieves the field's estimated, aggregated survival rate and the number of required 
            # picks for given week from the equity table
            field_survival, num_picks = equity_ctx["field_survival"].get(w, (prob_bounds, 1))
            # Retrieves estimated pick percentage for team for given week from the equity table
            f_X = equity_ctx["pick_shares"].get((w, row.team), 0.0)
            # Adjusts dilution factor to account for entries that picked the same team as ours.
            # Needs to assume they also survived.
            dilution = max((field_survival ** num_picks) + f_X * (1 - prob_bounds), 1e-9)
            # Factors in dilution to the utility
            utility -= math.log(dilution)

        # Fills the cost matrix in with the utility value, but negates it since this is a minimization problem
        for r_i in week_index[w]:
            cost[r_i, t_i] = -utility  

    return cost, week_rows, teams

## Solve Cost Matrix

In [90]:
def solve(odds_df, state, mode="survival", pick_pct_df=None, availability_df=None,
          pool_size=None, herding_strength=8.0):
    equity_ctx = None
    if mode == "equity":
        equity_ctx = build_equity_context(odds_df, state, pick_pct_df, availability_df, pool_size, herding_strength)

    cost, week_slots, teams = build_cost_matrix(odds_df, state, mode, equity_ctx)
    row_idx, col_idx = linear_sum_assignment(cost)

    prob_lookup = odds_df.set_index(["week", "team"])["win_prob"]
    opp_lookup = odds_df.set_index(["week", "team"])["opponent"]

    picks = []
    unresolved = []

    for r, c in zip(row_idx, col_idx):
        week, team = week_slots[r], teams[c]
        if cost[r, c] >= 1e6 / 2:
            unresolved.append(week)
            continue
        picks.append({"week": week, "team": team, "win_prob": float(prob_lookup.get((week, team)))})

    picks.sort(key=lambda x: (x["week"], x["team"]))
    survival_prob = math.exp(sum(math.log(p["win_prob"]) for p in picks)) if picks else 0.0

    # Safety check (flagged, not auto-fixed): a double-pick week where both
    # chosen teams are each other's opponent can never actually survive --
    # at most one of them wins. This should be rare (the joint probability
    # of two teams from the same game both winning is usually far from
    # optimal), but it's cheap to catch explicitly rather than assume.
    conflicts = []
    by_week = {}
    for p in picks:
        by_week.setdefault(p["week"], []).append(p["team"])
    for week, teams_this_week in by_week.items():
        if len(teams_this_week) < 2:
            continue
        for i in range(len(teams_this_week)):
            for j in range(i + 1, len(teams_this_week)):
                t1, t2 = teams_this_week[i], teams_this_week[j]
                if opp_lookup.get((week, t1)) == t2:
                    conflicts.append((week, t1, t2))

    return picks, survival_prob, sorted(set(unresolved)), conflicts, equity_ctx

## Display helper

In [91]:
def print_plan(picks, state, equity_ctx=None):
    by_week = {}
    for p in picks:
        by_week.setdefault(p["week"], []).append(p)

    print(f"{'Week':<6}{'Team':<8}{'Win %':<8}")
    print("-" * 22)
    for week in sorted(by_week):
        entries = sorted(by_week[week], key=lambda e: -e["win_prob"])
        is_current = (week == state["current_week"])
        for e in entries:
            marker = ""
            if is_current:
                marker = "  <-- THIS WEEK'S PICK" + (" (double week)" if len(entries) > 1 else "")
            print(f"{week:<6}{e['team']:<8}{e['win_prob']*100:>5.1f}%{marker}")
        if len(entries) > 1:
            combined = 1.0
            for e in entries:
                combined *= e["win_prob"]
            print(f"         (combined win prob for week {week}: {combined*100:.1f}%)")

    if equity_ctx and "pool_sizes" in equity_ctx:
        print(f"\n{'Week':<6}{'Proj. pool size':<18}{'Field survival rate':<22}")
        print("-" * 46)
        for week in sorted(equity_ctx["pool_sizes"]):
            N_w = equity_ctx["pool_sizes"][week]
            S, K = equity_ctx["field_survival"].get(week, (None, None))
            field_rate = f"{(S**K)*100:.1f}%" if S is not None else "n/a"
            print(f"{week:<6}{N_w:<18.1f}{field_rate:<22}")
        print("(projected forward from your current pool-size input using modeled "
              "pick-share data -- treat later weeks as estimates, not facts. Informational "
              "only -- doesn't change picks; see the equity-mode notes for why.)")

## Config

- `POOL_NAME`: which pool this run is for
- `LOCK`: leave as `None` to just see recommendations. **After** you've made this week's real pick(s) elsewhere, set this (e.g. `"BUF"` or `"BUF,KC"` for a double-pick week) and re-run the notebook once to record it and advance state -- then set it back to `None` for next time.
- `POOL_SIZE`: optional, informational only -- prints a projected pool-size trajectory but doesn't affect picks (see the equity-mode notes above for why).

In [ ]:
ODDS_CSV = "odds.csv"        # path to your odds CSV
STATE_FILE = "survivor_state.json"  # one shared file -- can hold multiple pools
POOL_NAME = "pool_1"               # which pool this run is for -- use a different name per pool
MODE = "equity"                   # "survival" or "equity"
PICKS_CSV = "projected_pick_distribution.csv"                    # path to pick-popularity CSV -- optional, fills gaps automatically
AVAILABILITY_CSV = "pool_1_availability.csv"             # path to availability CSV -- optional, improves the fallback estimate
POOL_SIZE = 15000                   # current remaining entries in your pool -- informational only
HERDING_STRENGTH = 8.0              # how sharply the fallback pick-share estimate favors chalk (equity mode) -- see herding_calibration.ipynb for how this default was derived
DOUBLE_PICK_WEEKS = [3,6,9,12,13,14,15,16]                   # e.g. [12, 15] -- weeks needing 2 picks. Set once; persists in state.
LOCK = None                         # e.g. "BUF" or "BUF,KC" -- set AFTER making this week's real pick, then re-run

## Run

In [93]:
odds_df = load_odds(ODDS_CSV)
state = load_state(STATE_FILE, POOL_NAME)

if DOUBLE_PICK_WEEKS:
    state["double_pick_weeks"] = sorted(set(DOUBLE_PICK_WEEKS))
    save_state(STATE_FILE, POOL_NAME, state)

if LOCK:
    teams_to_lock = [t.strip().upper() for t in LOCK.split(",") if t.strip()]
    locked_week = state["current_week"]
    is_double = locked_week in state["double_pick_weeks"]
    expected = 2 if is_double else 1
    if len(teams_to_lock) != expected:
        raise ValueError(f"Week {locked_week} is a {'double' if is_double else 'single'}-pick week: "
                          f"expected {expected} team(s), got {len(teams_to_lock)} ({teams_to_lock}).")
    for t in teams_to_lock:
        if t in state["used_teams"]:
            raise ValueError(f"{t} has already been used this season.")
    state["used_teams"].extend(teams_to_lock)
    state["current_week"] += 1
    save_state(STATE_FILE, POOL_NAME, state)
    print(f"Locked in {', '.join(teams_to_lock)} for week {locked_week} (pool: {POOL_NAME}). State saved to {STATE_FILE}.")
    print(f"Used teams so far: {state['used_teams']}")
    print("\nSet LOCK back to None above before your next run.")
else:
    pick_pct_df = load_pick_dist(PICKS_CSV) if PICKS_CSV else None
    availability_df = load_team_availability(AVAILABILITY_CSV) if AVAILABILITY_CSV else None
    if MODE == "equity" and pick_pct_df is None:
        print("[!] MODE='equity' with no PICKS_CSV: every team's pick share is being estimated by the "
              "fallback model alone. For real accuracy, supply actual public pick percentages where you have them.\n")

    picks, survival_prob, unresolved, conflicts, equity_ctx = solve(
        odds_df, state, MODE, pick_pct_df, availability_df, POOL_SIZE, HERDING_STRENGTH)

    print(f"Pool: {POOL_NAME}  |  Mode: {MODE}  |  Solving from week {state['current_week']}  |  "
          f"Teams already used: {state['used_teams'] or 'none'}  |  "
          f"Double-pick weeks: {state['double_pick_weeks'] or 'none'}\n")

    print_plan(picks, state, equity_ctx)

    if unresolved:
        print(f"\n[!] Could not find a feasible team for week(s): {unresolved} "
              f"(likely ran out of unused teams playing that week -- check your odds file coverage).")

    for week, t1, t2 in conflicts:
        print(f"\n[!] Week {week}: recommended picks {t1} and {t2} play EACH OTHER -- at most one can win, "
              f"so this pairing can never actually survive. This should be rare; consider locking one of "
              f"them to a different team by hand.")

    print(f"\nProjected probability of surviving the remaining season: {survival_prob*100:.2f}%")

    this_week_picks = [p for p in picks if p["week"] == state["current_week"]]
    if this_week_picks:
        names = ", ".join(f"{p['team']} ({p['win_prob']*100:.1f}%)" for p in this_week_picks)
        lock_val = ",".join(p["team"] for p in this_week_picks)
        print(f"\n>>> Recommended pick(s) for week {state['current_week']}: {names}")
        print(f"Once you've made this pick for real, set LOCK = \"{lock_val}\" above and re-run this notebook.")

 week team  pick_share
    7  DEN    0.274648
    7  LAR    0.136547
    7  HOU    0.130894
    7  TEN    0.066634
    7  NYJ    0.051332
    7  SEA    0.047794
    7   SF    0.046115
    7  PHI    0.039425
    7  BAL    0.039200
    7  MIN    0.038049
    7  PIT    0.027285
    7  DET    0.023286
    7  CHI    0.020491
    7   TB    0.020446
    7  CAR    0.006950
    7  MIA    0.005032
    7   NO    0.004934
    7  IND    0.003797
    7   NE    0.003523
    7  CLE    0.003202
    7   GB    0.002708
    7  DAL    0.002509
    7   KC    0.001657
    7  CIN    0.001587
    7  ATL    0.001217
    7  NYG    0.000454
    7  ARI    0.000183
    7   LV    0.000101
Pool: splash_25  |  Mode: equity  |  Solving from week 2  |  Teams already used: ['PIT']  |  Double-pick weeks: [3, 6, 9, 12, 13, 14, 15, 16]

Week  Team    Win %   
----------------------
2     TB       76.6%  <-- THIS WEEK'S PICK
3     SF       79.1%
3     NYG      68.1%
         (combined win prob for week 3: 53.9%)
4     MIN   